In [130]:
# import libraries

import pandas  as pd
import numpy as np
import matplotlib.pyplot as mp
import seaborn as sb 
import csv 


In [131]:
df = pd.read_csv('cleaned_retail.csv',dtype={'Invoice': str})


In [132]:
df.head()


,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,83.40
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,100.80
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,30.00


In [133]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1041670 entries, 0 to 1041669
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Unnamed: 0   1041670 non-null  int64  
 1   Invoice      1041670 non-null  str    
 2   StockCode    1041670 non-null  str    
 3   Description  1041670 non-null  str    
 4   Quantity     1041670 non-null  int64  
 5   InvoiceDate  1041670 non-null  str    
 6   Price        1041670 non-null  float64
 7   Customer ID  805549 non-null   float64
 8   Country      1041670 non-null  str    
 9   Revenue      1041670 non-null  float64
dtypes: float64(3), int64(2), str(5)
memory usage: 149.3 MB


In [163]:
df.head()
df.shape

(1041670, 11)

# Total SKUs (4,746) — The total number of different products being tracked

In [135]:
# Remove Trailing Space
with_trail_spaces = df['StockCode'].copy()

without_trail_space = (df['StockCode'].astype(str).str.strip().str.upper())

print(f'Unique values before cleaning: {with_trail_spaces.nunique()}')
print(f'Unique values after cleaning: {without_trail_space.nunique()}')


Unique values before cleaning: 4918
Unique values after cleaning: 4746


In [161]:
# Apply cleaning permanently to the original column
df['StockCode'] = without_trail_space
print(len(df))

1041670


In [137]:
# Extract Date 
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df['Date']=df['InvoiceDate'].dt.date
df.head()


,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Date
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,83.40,2009-12-01
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00,2009-12-01
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00,2009-12-01
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,100.80,2009-12-01
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,30.00,2009-12-01


# Total quantity sold per SKU per day

In [ ]:

daily_demand = df.groupby(['StockCode','Description','Date'])['Quantity'].sum().reset_index()
daily_demand.rename(columns={'Quantity':'DailyQty'},inplace='True')
print(daily_demand.shape)
print(daily_demand.head(10))
daily_demand.to_csv("daily_demand.csv",index=False)


(533346, 4)
  StockCode                  Description       Date  DailyQty
0     10002  INFLATABLE POLITICAL GLOBE  2009-12-01        12
1     10002  INFLATABLE POLITICAL GLOBE  2009-12-03         7
2     10002  INFLATABLE POLITICAL GLOBE  2009-12-04        73
3     10002  INFLATABLE POLITICAL GLOBE  2009-12-06        49
4     10002  INFLATABLE POLITICAL GLOBE  2009-12-07         2
5     10002  INFLATABLE POLITICAL GLOBE  2009-12-08        12
6     10002  INFLATABLE POLITICAL GLOBE  2009-12-10         1
7     10002  INFLATABLE POLITICAL GLOBE  2009-12-11         9
8     10002  INFLATABLE POLITICAL GLOBE  2009-12-14        37
9     10002  INFLATABLE POLITICAL GLOBE  2009-12-21        12
533346


In [139]:
df_2 = pd.read_csv('daily_demand.csv',dtype={'StockCode': str})
df_2.head()
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 533346 entries, 0 to 533345
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   StockCode    533346 non-null  str  
 1   Description  533346 non-null  str  
 2   Date         533346 non-null  str  
 3   DailyQty     533346 non-null  int64
dtypes: int64(1), str(3)
memory usage: 37.6 MB


# How many days of dataset

In [ ]:

df['Date']=pd.to_datetime(df['Date'])

full_date_range=pd.date_range(df_2['Date'].min(),df_2['Date'].max())
total_days = len(full_date_range)
print(f"Total days in dataset : {total_days}")


Total days in dataset : 739


# For each SKU , total quantity sold and count of days  it appeared

In [ ]:

sku_stats=df_2.groupby(['StockCode','Description']).agg(
    TotalQtySold =('DailyQty','sum'),
    DaysWithSales=('DailyQty','count'),
    StdDevOnSaleDays=('DailyQty','std')
).reset_index()

print(sku_stats)


     StockCode                  Description  TotalQtySold  DaysWithSales  \
0        10002  INFLATABLE POLITICAL GLOBE           8836            225   
1       10002R        ROBOT PENCIL SHARPNER             4              3   
2        10080     GROOVY CACTUS INFLATABLE           315             26   
3        10109         BENDY COLOUR PENCILS             4              1   
4        10120                 DOGGY RUBBER           680             64   
...        ...                          ...           ...            ...   
5427      POST                      POSTAGE          5461            514   
5428         S                      SAMPLES             3              3   
5429    SP1002       KID'S CHALKBOARD/EASEL             5              1   
5430   TEST001      This is a test product.            50              6   
5431   TEST002      This is a test product.             1              1   

      StdDevOnSaleDays  
0                93.09  
1                 0.58  
2           

# Average daily demand : how many units of a product typically sell per day

In [ ]:

sku_stats['AvgDailyDemand']=sku_stats['TotalQtySold'] / total_days
sku_stats['AvgDailyDemand_ActiveDays'] = sku_stats['TotalQtySold'] / sku_stats['DaysWithSales']

In [143]:

sku_stats['StdDevOnSaleDays']=sku_stats['StdDevOnSaleDays'].fillna(0)

In [144]:
# print(sku_stats)
print(sku_stats.shape)
print(sku_stats.sort_values(by='TotalQtySold',ascending=False).head(10))
sku_stats.to_csv('sku_stats.csv',index=False)

(5432, 7)
     StockCode                         Description  TotalQtySold  \
4103     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        110138   
4826    85123A  WHITE HANGING HEART T-LIGHT HOLDER         96677   
4529     84879       ASSORTED COLOUR BIRD ORNAMENT         81809   
3349     23843         PAPER CRAFT , LITTLE BIRDIE         80995   
4799    85099B             JUMBO BAG RED RETROSPOT         79279   
2804     23166      MEDIUM CERAMIC TOP STORAGE JAR         78033   
117      17003                 BROCADE RING PURSE          71430   
1378     21977  PACK OF 60 PINK PAISLEY CAKE CASES         56794   
4668     84991         60 TEATIME FAIRY CAKE CASES         54716   
1601     22197                SMALL POPCORN HOLDER         49948   

      DaysWithSales  StdDevOnSaleDays  AvgDailyDemand  \
4103            470            527.83          149.04   
4826            602            219.12          130.82   
4529            582            243.66          110.70   
3349         

In [145]:
# Look at the raw data for this specific product to see why it has 80k+ quantity in 1 day
df_2[df_2['StockCode'] == '23843']
sku_stats[sku_stats['StockCode'] == '23843']
# 'Paper Craft, Little Birdie' appears to be a top product by volume, it is an extreme outlier driven entirely by a single wholesale transaction of 80,995 units on a single day."

,StockCode,Description,TotalQtySold,DaysWithSales,StdDevOnSaleDays,AvgDailyDemand,AvgDailyDemand_ActiveDays
3349,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,1,0.00,109.60,"80,995.00"


In [146]:
df_3 =  pd.read_csv("sku_stats.csv")
df_3.info()

<class 'pandas.DataFrame'>
RangeIndex: 5432 entries, 0 to 5431
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StockCode                  5432 non-null   str    
 1   Description                5432 non-null   str    
 2   TotalQtySold               5432 non-null   int64  
 3   DaysWithSales              5432 non-null   int64  
 4   StdDevOnSaleDays           5432 non-null   float64
 5   AvgDailyDemand             5432 non-null   float64
 6   AvgDailyDemand_ActiveDays  5432 non-null   float64
dtypes: float64(3), int64(2), str(2)
memory usage: 469.5 KB


# Safety Stock : Extra backup stock kept aside in case demand suddenly increases

In [147]:
lead_time_days = 7
service_level_z = 1.65
# Formula : Safety Stock = Z * StdDev(demand) * sqrt(Lead Time)
df_3['Safety_Stock'] = service_level_z * df_3['StdDevOnSaleDays'] * np.sqrt(lead_time_days)
df_3['Safety_Stock']=df_3['Safety_Stock'].round(0)

# Reorder Point — The stock level at which a new order should be placed

In [148]:
# formula : reorder point = ( avg daily demand * lead time) + safety stock
df_3['ReorderPoint'] = (df_3['AvgDailyDemand'] * lead_time_days) + df_3['Safety_Stock']
df_3['ReorderPoint']=df_3['ReorderPoint'].round(0)
print(df_3.sort_values(by='TotalQtySold', ascending=False).head(10))
df_3.to_csv("sku_reorder_point.csv",index=False)

     StockCode                         Description  TotalQtySold  \
4103     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        110138   
4826    85123A  WHITE HANGING HEART T-LIGHT HOLDER         96677   
4529     84879       ASSORTED COLOUR BIRD ORNAMENT         81809   
3349     23843         PAPER CRAFT , LITTLE BIRDIE         80995   
4799    85099B             JUMBO BAG RED RETROSPOT         79279   
2804     23166      MEDIUM CERAMIC TOP STORAGE JAR         78033   
117      17003                 BROCADE RING PURSE          71430   
1378     21977  PACK OF 60 PINK PAISLEY CAKE CASES         56794   
4668     84991         60 TEATIME FAIRY CAKE CASES         54716   
1601     22197                SMALL POPCORN HOLDER         49948   

      DaysWithSales  StdDevOnSaleDays  AvgDailyDemand  \
4103            470            527.83          149.04   
4826            602            219.12          130.82   
4529            582            243.66          110.70   
3349              1    

In [149]:
df_4=pd.read_csv('sku_reorder_point.csv')
df_4.info()

<class 'pandas.DataFrame'>
RangeIndex: 5432 entries, 0 to 5431
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StockCode                  5432 non-null   str    
 1   Description                5432 non-null   str    
 2   TotalQtySold               5432 non-null   int64  
 3   DaysWithSales              5432 non-null   int64  
 4   StdDevOnSaleDays           5432 non-null   float64
 5   AvgDailyDemand             5432 non-null   float64
 6   AvgDailyDemand_ActiveDays  5432 non-null   float64
 7   Safety_Stock               5432 non-null   float64
 8   ReorderPoint               5432 non-null   float64
dtypes: float64(5), int64(2), str(2)
memory usage: 554.4 KB


#This dataset only has historical sales — it doesn't tell us what's sitting in the warehouse today. Real companies pull this from their inventory system, so here we simulate it realistically and document it clearly as an assumption.

In [150]:
# Asssumptions based Current Stock
np.random.seed(42)
df_4['SimulatedDaysOfStock']=np.random.randint(1,20,size=len(df_4))
df_4['CurrentStock'] = df_4['AvgDailyDemand'] * df_4['SimulatedDaysOfStock']

In [151]:
# Reorder flag 
df_4['ReorderFlag'] = np.where(df_4['CurrentStock'] < df_4['ReorderPoint'],'Reorder Now','OK')
print(df_4['ReorderFlag'].value_counts())
print(df_4.sort_values(by='TotalQtySold',ascending=False).head())
df_4.to_csv('final_inventory_analysis.csv',index=False)

ReorderFlag
Reorder Now    5084
OK              348
Name: count, dtype: int64
     StockCode                         Description  TotalQtySold  \
4103     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        110138   
4826    85123A  WHITE HANGING HEART T-LIGHT HOLDER         96677   
4529     84879       ASSORTED COLOUR BIRD ORNAMENT         81809   
3349     23843         PAPER CRAFT , LITTLE BIRDIE         80995   
4799    85099B             JUMBO BAG RED RETROSPOT         79279   

      DaysWithSales  StdDevOnSaleDays  AvgDailyDemand  \
4103            470            527.83          149.04   
4826            602            219.12          130.82   
4529            582            243.66          110.70   
3349              1              0.00          109.60   
4799            461            198.30          107.28   

      AvgDailyDemand_ActiveDays  Safety_Stock  ReorderPoint  \
4103                     234.34      2,304.00      3,347.00   
4826                     160.59        957.0

In [152]:
df_5 = pd.read_csv('final_inventory_analysis.csv')
df_5.info()

<class 'pandas.DataFrame'>
RangeIndex: 5432 entries, 0 to 5431
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StockCode                  5432 non-null   str    
 1   Description                5432 non-null   str    
 2   TotalQtySold               5432 non-null   int64  
 3   DaysWithSales              5432 non-null   int64  
 4   StdDevOnSaleDays           5432 non-null   float64
 5   AvgDailyDemand             5432 non-null   float64
 6   AvgDailyDemand_ActiveDays  5432 non-null   float64
 7   Safety_Stock               5432 non-null   float64
 8   ReorderPoint               5432 non-null   float64
 9   SimulatedDaysOfStock       5432 non-null   int64  
 10  CurrentStock               5432 non-null   float64
 11  ReorderFlag                5432 non-null   str    
dtypes: float64(6), int64(3), str(3)
memory usage: 737.0 KB


# Total revenue per SKU

In [ ]:

transactions = pd.read_csv('cleaned_retail.csv',dtype={'Invoice' : str})
sku_revenue=transactions.groupby('StockCode')['Revenue'].sum().reset_index()
sku_revenue.rename(columns={'Revenue':'TotalRevenue'},inplace=True)
print(pd.pivot_table(sku_revenue,index='StockCode',values='TotalRevenue'))

              TotalRevenue
StockCode                 
10002             7,097.90
10002R               20.57
10080               129.29
10109                 1.68
10120               146.10
...                    ...
gift_0001_40        166.09
gift_0001_50        253.59
gift_0001_70         59.57
gift_0001_80         69.56
m                    15.05

[4918 rows x 1 columns]


In [154]:
# Merge revenue with main dataframe
df_5=df_5.merge(sku_revenue,on='StockCode',how='left')
df_5=df_5.sort_values(by='TotalRevenue',ascending=False).reset_index(drop=True)


In [155]:
# df_5.drop(columns=['TotalRevenue_y','TotalRevenue_x'],inplace=True)

# Cumulative % of Total Revenue
df_5['CumulativeRevenue'] = df_5['TotalRevenue'].cumsum()
df_5['CumulativeRevenuePercent']=df_5['CumulativeRevenue'] / df_5['TotalRevenue'].sum() * 100



In [156]:
# Assign ABC category based on Total Revenue or  Pareto Analysis 80/20 rule 
def assign_abc(percent):
    if percent <=80:
        return 'A'
    elif percent <=95 :
        return 'B'
    else :
        return 'C' 

df_5['ABC_Category']=df_5['CumulativeRevenuePercent'].apply(assign_abc)

In [167]:
# Summary
pd.options.display.float_format = '{:,.2f}'.format
print(f'Total Revenue : {df_5['TotalRevenue'].sum()}')
print(df_5['ABC_Category'].value_counts())
print(df_5.groupby('ABC_Category')['TotalRevenue'].sum())
df_5.to_csv("final_inventory_with_abc.csv", index=False)

Total Revenue : 27475894.289
ABC_Category
C    2813
B    1462
A    1157
Name: count, dtype: int64
ABC_Category
A   21,978,688.40
B    4,122,819.76
C    1,374,386.13
Name: TotalRevenue, dtype: float64


Overall revenue : £27.47M
Category A revenue (£21.97M from 1,157 products) — a small group of products brings in most of the money
Category B revenue (£4.12M from 1,462 products) — a mid-size group with moderate sales contribution
Category C revenue (£1.37M from 2,813 products) — most products individually contribute very little revenue

In [158]:
df_5.head()

,StockCode,Description,TotalQtySold,DaysWithSales,StdDevOnSaleDays,AvgDailyDemand,AvgDailyDemand_ActiveDays,Safety_Stock,ReorderPoint,SimulatedDaysOfStock,CurrentStock,ReorderFlag,TotalRevenue,CumulativeRevenue,CumulativeRevenuePercent,ABC_Category
0,22423,REGENCY CAKESTAND 3 TIER,27577,514,60.18,37.32,53.65,263.00,524.00,4,149.27,Reorder Now,"344,563.25","344,563.25",1.25,A
1,M,Manual,10055,378,129.32,13.61,26.60,565.00,660.00,1,13.61,Reorder Now,"340,716.28","685,279.53",2.49,A
2,DOT,DOTCOM POSTAGE,1436,441,3.70,1.94,3.26,16.00,30.00,1,1.94,Reorder Now,"322,657.48","1,007,937.01",3.67,A
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,96677,602,219.12,130.82,160.59,957.00,"1,873.00",12,"1,569.86",Reorder Now,"263,094.37","1,271,031.38",4.63,A
4,85123A,CREAM HANGING HEART T-LIGHT HOLDER,61,2,37.48,0.08,30.50,164.00,165.00,6,0.50,Reorder Now,"263,094.37","1,534,125.75",5.58,A
